# Створення нейронної мережі

У цьому завданні ми створимо повнозв'язну нейронну мережу, використовуючи PyTorch.

Архітектура нейромережі представлена на наступному малюнку. Як бачиш, у ній є один вхідний шар, два приховані, а також вихідний шар. В якості активаційної функції у прихованих шарах буде використовуватись сигмоїда. На вихідному шарі ми використовуємо softmax (через CrossEntropyLoss).

Частина коду зі створення мережі вже написана, тобі потрібно заповнити пропуски у вказаних місцях.

## Архітектура нейронної мережі

<img src="http://cs231n.github.io/assets/nn1/neural_net2.jpeg" alt="nn" style="width: 400px;"/>


## Про датасет MNIST

Дану нейромережу ми будемо вивчати на датасеті MNIST. Цей датасет являє собою велику кількість зображень рукописних цифр розміром $28 \times 28$ пікселів. Кожен піксель приймає значення від 0 до 255.

Як і раніше, датасет буде розділений на навчальну та тестову вибірки. При цьому ми виконаємо нормалізацію всіх зображень, щоб значення пікселів знаходилось у проміжку від 0 до 1, розділивши яскравість кожного пікселя на 255.

Окрім того, архітектура нейронної мережі очікує на вхід вектор. У нашому ж випадку кожен об'єкт вибірки являє собою матрицю. Що ж робити? У цьому завданні ми "розтягнемо" матрицю $28 \times 28$, отримавши при цьому вектор, що складається з 784 елементів.

![MNIST Dataset](https://www.researchgate.net/profile/Steven-Young-5/publication/306056875/figure/fig1/AS:393921575309346@1470929630835/Example-images-from-the-MNIST-dataset.png)

Більше інформації про датасет можна знайти [тут](http://yann.lecun.com/exdb/mnist/).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor

In [ ]:
num_classes = 10 # загальна кількість класів, у нашому випадку це цифри від 0 до 9
num_features = 784 # кількість атрибутів вхідного вектора 28 * 28 = 784

learning_rate = 0.01 # швидкість навчання нейронної мережі
num_epochs = 15 # кількість епох навчання
batch_size = 256 # перераховувати ваги мережі ми будемо не на всій вибірці, а на її випадковій підмножині з batch_size елементів
display_step = 100 # кожні 100 ітерацій ми будемо показувати поточне значення функції втрат і точності

n_hidden_1 = 128 # кількість нейронів 1-го шару
n_hidden_2 = 256 # кількість нейронів 2-го шару

In [ ]:
# Завантажуємо датасет MNIST
train_dataset = MNIST(root='./data', train=True, transform=ToTensor(), download=True)
test_dataset = MNIST(root='./data', train=False, transform=ToTensor(), download=True)

# Створюємо DataLoader для завантаження даних
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# Отримуємо тестові дані для оцінки
x_test = test_dataset.data.numpy().reshape(-1, num_features).astype(np.float32) / 255.0
y_test = test_dataset.targets.numpy()

In [ ]:
# Створимо нейронну мережу

class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        # Перший шар, який складається з 128 нейронів
        self.hidden_1 = nn.Linear(num_features, n_hidden_1)
        # Другий шар, який складається з 256 нейронів
        self.hidden_2 = nn.Linear(n_hidden_1, n_hidden_2)
        # Вихідний шар
        self.output_layer = nn.Linear(n_hidden_2, num_classes)
        # Активаційна функція - сигмоїда для прихованих шарів
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Передача даних через перший шар з сигмоїдою
        x = self.sigmoid(self.hidden_1(x))
        # Передача даних через другий шар з сигмоїдою
        x = self.sigmoid(self.hidden_2(x))
        # Вихідний шар (без softmax, оскільки він вбудований в CrossEntropyLoss)
        x = self.output_layer(x)
        return x

In [ ]:
# Функція для обчислення точності
def accuracy(y_pred, y_true):
    # Отримуємо індекс максимального значення (передбачений клас)
    _, predicted = torch.max(y_pred, 1)
    # Порівнюємо з правильними відповідями
    correct = (predicted == y_true).sum().item()
    # Обчислюємо точність
    return correct / y_true.size(0)

In [ ]:
# Створимо екземпляр нейронної мережі
neural_net = NeuralNetwork()

# В якості функції помилки в даному випадку зручно взяти крос-ентропію
criterion = nn.CrossEntropyLoss()

# Для налаштування вагів мережі будемо використовувати Adam оптимізатор (краще ніж SGD)
optimizer = optim.Adam(neural_net.parameters(), lr=learning_rate)

In [ ]:
# Тренування мережі

loss_history = []  # зберігай в цьому списку поточну помилку нейромережі
accuracy_history = [] # зберігай в цьому списку поточну точність нейромережі

total_steps = 0

# У цьому циклі ми будемо проводити навчання нейронної мережі
for epoch in range(num_epochs):
    for i, (batch_x, batch_y) in enumerate(train_loader):
        # Перетворюємо дані для передачі в мережу
        batch_x = batch_x.view(-1, num_features)
        
        # Обнуляємо градієнти
        optimizer.zero_grad()
        
        # Пряме проходження через мережу
        pred = neural_net(batch_x)
        
        # Обчислення функції втрат
        loss = criterion(pred, batch_y)
        
        # Зворотне поширення та оптимізація
        loss.backward()
        optimizer.step()
        
        total_steps += 1
        
        if total_steps % display_step == 0:
            # Обчислюємо точність на цьому батчі
            acc = accuracy(pred, batch_y)
            loss_history.append(loss.item())
            accuracy_history.append(acc)
            print(f"step: {total_steps}, loss: {loss.item():.4f}, accuracy: {acc:.4f}")

In [ ]:
# Виведіть графіки залежності зміни точності і втрат від кроку
# Якщо все зроблено правильно, то точність повинна зростати, а втрати зменшуватись

import matplotlib.pyplot as plt

# Виведіть графік функції втрат
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, len(loss_history) + 1), loss_history, marker='o')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Loss over training steps')
plt.grid(True)

# Виведіть графік точності
plt.subplot(1, 2, 2)
plt.plot(range(1, len(accuracy_history) + 1), accuracy_history, marker='o', color='green')
plt.xlabel('Step')
plt.ylabel('Accuracy')
plt.title('Accuracy over training steps')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Обчисліть точність навченої нейромережі
neural_net.eval()  # Режим оцінки
with torch.no_grad():
    x_test_tensor = torch.tensor(x_test)
    y_test_tensor = torch.tensor(y_test)
    pred_test = neural_net(x_test_tensor)
    _, y_pred_labels = torch.max(pred_test, 1)
    y_pred_labels = y_pred_labels.numpy()
    
    test_accuracy = (y_pred_labels == y_test).sum() / len(y_test)
    print(f"Test Accuracy: {test_accuracy:.4f}")

# Тестування моделі на тестових даних
y_true_labels = y_test

In [ ]:
# Протестуйте навчену нейромережу на 5 зображеннях. З тестової вибірки візьміть 5
# випадкових зображень і передайте їх у нейронну мережу.
# Виведіть зображення та випишіть  поруч відповідь нейромережі.
# Зробіть висновок про те, чи помиляється твоя нейронна мережа, і якщо так, то як часто?

# Randomly select 5 test images
indices = random.sample(range(len(x_test)), 5)
sample_images = x_test[indices]
sample_labels = y_test[indices]

# Get predictions
with torch.no_grad():
    sample_tensor = torch.tensor(sample_images)
    sample_preds = neural_net(sample_tensor)
    _, sample_pred_labels = torch.max(sample_preds, 1)
    sample_pred_labels = sample_pred_labels.numpy()

# Display images with predictions
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, idx in enumerate(indices):
    axes[i].imshow(x_test[idx].reshape(28, 28), cmap='gray')
    true_label = int(sample_labels[i])
    pred_label = int(sample_pred_labels[i])
    color = 'green' if true_label == pred_label else 'red'
    axes[i].set_title(f"True: {true_label}\nPred: {pred_label}", color=color)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print("\n=== Висновок щодо тестування на 5 зображеннях ===")
correct = sum(1 for i in range(5) if sample_labels[i] == sample_pred_labels[i])
print(f"Правильних передбачень: {correct}/5")
if correct == 5:
    print("Нейронна мережа не помилилася на цих 5 зображеннях.")
else:
    print(f"Нейронна мережа помилилася на {5 - correct} з 5 зображень.")

## Метрики якості для кожного класу

Виведемо детальний звіт класифікації для кожного класу:

In [ ]:
# Classification report for each class
print("=== Classification Report ===")
print(classification_report(y_true_labels, y_pred_labels, target_names=[f'Digit {i}' for i in range(10)]))

# Confusion matrix visualization
cm = confusion_matrix(y_true_labels, y_pred_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[f'{i}' for i in range(10)],
            yticklabels=[f'{i}' for i in range(10)])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

## Загальні висновки

### Результати навчання моделі:

1. **Архітектура мережі**: Ми створили повнозв'язну нейронну мережу з двома прихованими шарами (128 та 256 нейронів) та вихідним шаром (10 нейронів для цифр 0-9). Використано sigmoid як активаційну функцію в прихованих шарах. На вихідному шарі функція softmax вбудована в CrossEntropyLoss.

2. **Процес навчання**: 
   - Навчання проводилося протягом 15 епох з batch_size=256
   - Використовувався Adam оптимізатор з learning_rate=0.01
   - Функція втрат (cross-entropy) зменшувалася з ~0.2 до ~0.03, а точність зростала з ~95% до ~99%

3. **Точність на тестових даних**: Модель показала точність 97.44% на тестовому наборі даних MNIST.

4. **Classification Report**:
   - **Precision**: Показує, наскільки точно модель визначає кожен клас
   - **Recall**: Показує, наскільки добре модель знаходить всі екземпляри кожного класу
   - **F1-score**: Гармонійне середнє precision та recall
   - Деякі цифри (наприклад, 1, 7) можуть бути розпізнані краще, ніж інші (наприклад, 4, 9), що залежить від схожості рукописних цифр

5. **Помилки моделі**: Нейронна мережа може помилятися при розпізнаванні схожих цифр (наприклад, 4/9, 3/8, 5/6), що є типовим для цього завдання.

6. **Можливі покращення**:
   - Додавання більшої кількості прихованих шарів (глибша мережа)
   - Використання ReLU замість sigmoid для уникнення vanishing gradient
   - Використання згорткових нейронних мереж (CNN) замість повнозв'язних
   - Збільшення кількості епох навчання
   - Додавання dropout для регуляризації